# SarcasmIQ - AI-Powered Sarcasm Detection
## Capstone Project 2 | NLP Track

**Author:** Simran Rani | Data Analyst and Aspiring Data Scientist

**Objective:** Build a complete end-to-end NLP pipeline to detect whether a news headline is sarcastic or not. We use an 80-20 train-test split and examine model performance on both training and test samples.

**What we will cover in this notebook:**
- Step 1: Load and clean the dataset
- Step 2: Explore the data visually (EDA)
- Step 3: Classical Machine Learning with TF-IDF
- Step 4: Hyperparameter Tuning and Ensemble Models
- Step 5: Deep Learning with RNN, GRU, LSTM and BiLSTM
- Step 6: Pre-trained GloVe Embeddings for best accuracy
- Step 7: Save models and test on new headlines

**Dataset:** News Headlines Dataset for Sarcasm Detection (28,619 headlines)

**Best Result Achieved:** 88.9% accuracy using GloVe + Stacked Bidirectional LSTM


## Section 1: Install and Import All Libraries

We import everything we need upfront so there are no interruptions later. This includes libraries for data handling, visualization, text processing, classical machine learning, and deep learning.


In [1]:
# Standard Python libraries
import json
import re
import os
import pickle
import warnings
import urllib.request
import zipfile

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# Natural Language Processing
import nltk
for pkg in ["stopwords", "wordnet", "omw-1.4"]:
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Classical Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, SimpleRNN, GRU, LSTM, Bidirectional,
    Dense, Dropout, SpatialDropout1D, BatchNormalization
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")


TensorFlow version: 2.17.0
All libraries imported successfully!


## Section 2: Load the Dataset

We use the News Headlines Dataset for Sarcasm Detection. It is a popular NLP benchmark dataset where each record contains:
- headline: the news headline text
- is_sarcastic: 1 if the headline is sarcastic, 0 if it is not
- article_link: the source URL (we will not use this for modelling)

We download the dataset directly from GitHub so the notebook is fully self-contained and can run on any machine.


In [2]:
DATA_DIR  = "data"
DATA_PATH = os.path.join(DATA_DIR, "Sarcasm_Headlines_Dataset.json")
os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(DATA_PATH):
    print("Downloading dataset from GitHub...")
    url = "https://raw.githubusercontent.com/rishabhmisra/News-Headlines-Dataset-For-Sarcasm-Detection/master/Sarcasm_Headlines_Dataset.json"
    urllib.request.urlretrieve(url, DATA_PATH)
    print("Download complete!")
else:
    print("Dataset already exists locally. Skipping download.")

# The file is in JSON Lines format, meaning each line is a separate JSON object
records = [json.loads(line) for line in open(DATA_PATH, "r")]
df = pd.DataFrame(records)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Dataset already exists locally. Skipping download.
Dataset shape: (28619, 3)
Columns: ['is_sarcastic', 'headline', 'article_link']


,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [3]:
# Check how many sarcastic vs non-sarcastic headlines we have
print("Class Distribution:")
print(df["is_sarcastic"].value_counts())
print()
print("Sarcastic headlines    :", round(df["is_sarcastic"].mean() * 100, 1), "%")
print("Not Sarcastic headlines:", round((1 - df["is_sarcastic"].mean()) * 100, 1), "%")
print("Total records          :", len(df))


Class Distribution:
is_sarcastic
0    14985
1    13634
Name: count, dtype: int64

Sarcastic headlines    : 47.6 %
Not Sarcastic headlines: 52.4 %
Total records          : 28619


## Section 3: Data Cleansing

Before building any model, we need to clean the data carefully. Raw text contains URLs, punctuation, uppercase letters, and filler words that add noise without adding meaning.

Our cleaning steps:
1. Check for missing values and duplicates
2. Remove duplicate headlines
3. Convert all text to lowercase
4. Remove URLs and punctuation
5. Remove stopwords (common words like "the", "is", "a" that do not help identify sarcasm)
6. Apply lemmatization, which converts words to their base form. For example, "running" becomes "run" and "better" becomes "good"

Note: For deep learning with GloVe embeddings later, we will use a lighter cleaning approach that keeps stopwords, because phrases like "is, in fact" and "what a surprise" are actually important sarcasm signals.


In [4]:
# Check for missing values and duplicates
print("Missing Values:")
print(df.isnull().sum())
print()
print("Duplicate headlines:", df["headline"].duplicated().sum())
print()
print("Sample headlines from the dataset:")
for h in df["headline"].sample(5, random_state=42).values:
    print(" -", h)


Missing Values:
is_sarcastic    0
headline        0
article_link    0
dtype: int64

Duplicate headlines: 116

Sample headlines from the dataset:
 - states slow to shut down weak teacher education programs
 - drone places fresh kill on steps of white house
 - report: majority of instances of people getting lives back on track occur immediately after visit to buffalo wild wings
 - sole remaining lung filled with rich, satisfying flavor
 - the gop's stockholm syndrome


In [5]:
# Remove duplicates and rows with missing values
df = df.drop_duplicates(subset="headline").reset_index(drop=True)
df = df.dropna(subset=["headline", "is_sarcastic"]).reset_index(drop=True)
print("Rows after removing duplicates and nulls:", len(df))


Rows after removing duplicates and nulls: 28503


In [6]:
# Define the text cleaning function for classical ML
STOPWORDS  = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove anything that is not a letter or space
    text = re.sub(r"[^a-z\s]", " ", text)
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    # Split into words, remove stopwords, apply lemmatization
    tokens = text.split()
    tokens = [LEMMATIZER.lemmatize(w) for w in tokens if w not in STOPWORDS and len(w) > 1]
    return " ".join(tokens)

# Apply cleaning to all headlines
df["clean_headline"] = df["headline"].apply(clean_text)

# Remove any rows that became empty after cleaning
df = df[df["clean_headline"].str.len() > 0].reset_index(drop=True)

print("Total headlines after cleaning:", len(df))
print()
print("Examples showing before and after cleaning:")
for i in [0, 100, 500]:
    print("BEFORE:", df["headline"].iloc[i])
    print("AFTER :", df["clean_headline"].iloc[i])
    print()


Total headlines after cleaning: 28501

Examples showing before and after cleaning:
BEFORE: thirtysomething scientists unveil doomsday clock of hair loss
AFTER : thirtysomething scientist unveil doomsday clock hair loss

BEFORE: report: 70% of trump endorsements made after staring at bedroom ceiling for 4 hours
AFTER : report trump endorsement made staring bedroom ceiling hour

BEFORE: apple will probably introduce a new iphone sept. 9
AFTER : apple probably introduce new iphone sept



## Section 4: Exploratory Data Analysis

Now that the data is clean, we explore it visually to understand the patterns. Good EDA helps us make better decisions about features and models.

We will look at:
- Are the two classes balanced?
- Are sarcastic headlines longer or shorter than non-sarcastic ones?
- What words appear most often in each class?
- Is there any skewness in the length features that needs transformation?


In [7]:
# Add text length features for analysis
df["char_count"]       = df["headline"].str.len()
df["word_count"]       = df["headline"].str.split().apply(len)
df["clean_word_count"] = df["clean_headline"].str.split().apply(len)

print("Summary Statistics:")
df[["char_count", "word_count", "clean_word_count"]].describe().round(2)


Summary Statistics:


,char_count,word_count,clean_word_count
count,28501.00,28501.00,28501.00
mean,62.38,10.06,7.16
std,20.70,3.39,2.44
min,7.00,2.00,1.00
25%,49.00,8.00,6.00
50%,62.00,10.00,7.00
75%,75.00,12.00,9.00
max,926.00,151.00,107.00


In [8]:
# Class Distribution - how many sarcastic vs not sarcastic
counts = df["is_sarcastic"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(["Not Sarcastic", "Sarcastic"], counts.values,
            color=["#4F46E5", "#7C3AED"], edgecolor="white", width=0.5)
axes[0].set_title("Class Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Number of Headlines")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v) + " (" + str(round(v/len(df)*100, 1)) + "%)",
                 ha="center", fontweight="bold")

axes[1].pie(counts.values, labels=["Not Sarcastic", "Sarcastic"],
            autopct="%1.1f%%", colors=["#4F46E5", "#7C3AED"],
            startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Class Balance", fontsize=14, fontweight="bold")

plt.suptitle("The dataset is well balanced. No oversampling or undersampling is needed.",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig("class_distribution.png", bbox_inches="tight", dpi=150)
plt.show()


In [9]:
# Check skewness in headline length features
print("Skewness before log transformation:")
print("  Character count skewness:", round(df["char_count"].skew(), 3))
print("  Word count skewness      :", round(df["word_count"].skew(), 3))

# Apply log transformation to reduce skewness
df["char_count_log"] = np.log1p(df["char_count"])
df["word_count_log"] = np.log1p(df["word_count"])

print()
print("Skewness after log transformation:")
print("  Character count skewness:", round(df["char_count_log"].skew(), 3))
print("  Word count skewness      :", round(df["word_count_log"].skew(), 3))

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_info = [
    ("char_count",     "Character Count - Original"),
    ("char_count_log", "Character Count - After Log Transform"),
    ("word_count",     "Word Count - Original"),
    ("word_count_log", "Word Count - After Log Transform"),
]

for i, (col, title) in enumerate(plot_info):
    row_idx  = i // 2
    col_idx  = i % 2
    for label, color, name in [(0, "#4F46E5", "Not Sarcastic"), (1, "#7C3AED", "Sarcastic")]:
        axes[row_idx][col_idx].hist(
            df[df["is_sarcastic"] == label][col],
            bins=40, alpha=0.6, color=color, label=name
        )
    axes[row_idx][col_idx].set_title(title, fontweight="bold")
    axes[row_idx][col_idx].legend()
    axes[row_idx][col_idx].set_ylabel("Frequency")

plt.suptitle("Feature Distributions Before and After Log Transformation", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("distributions.png", bbox_inches="tight", dpi=150)
plt.show()


Skewness before log transformation:
  Character count skewness: 2.916
  Word count skewness      : 2.898

Skewness after log transformation:
  Character count skewness: -0.861
  Word count skewness      : -0.657


In [10]:
# Box plot - word count comparison between sarcastic and non-sarcastic
plt.figure(figsize=(7, 4))
sns.boxplot(x="is_sarcastic", y="word_count", data=df,
            palette=["#4F46E5", "#7C3AED"])
plt.xticks([0, 1], ["Not Sarcastic", "Sarcastic"])
plt.title("Word Count Distribution by Class", fontsize=13, fontweight="bold")
plt.xlabel("Class")
plt.ylabel("Number of Words")
plt.tight_layout()
plt.savefig("word_count_by_class.png", bbox_inches="tight", dpi=150)
plt.show()
print("Observation: Sarcastic headlines tend to be slightly longer on average.")
print("They often use more elaborate phrasing to set up the punchline.")


Observation: Sarcastic headlines tend to be slightly longer on average.
They often use more elaborate phrasing to set up the punchline.


In [11]:
# Word clouds - most common words in each class
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, label, title, colormap in zip(
    axes,
    [0, 1],
    ["Not Sarcastic Headlines", "Sarcastic Headlines"],
    ["Blues", "Purples"]
):
    text = " ".join(df[df["is_sarcastic"] == label]["clean_headline"])
    wc = WordCloud(width=700, height=400, background_color="white",
                   colormap=colormap, max_words=100).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title, fontsize=14, fontweight="bold")

plt.suptitle("Most Common Words in Each Class", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("word_clouds.png", bbox_inches="tight", dpi=150)
plt.show()
print("Observation: Sarcastic headlines use words like area, man, nation, report.")
print("These are typical of The Onion-style satirical news writing.")


Observation: Sarcastic headlines use words like area, man, nation, report.
These are typical of The Onion-style satirical news writing.


In [12]:
# Top 15 most frequent words in each class
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, label, title, color in zip(
    axes,
    [0, 1],
    ["Top 15 Words in Not Sarcastic Headlines", "Top 15 Words in Sarcastic Headlines"],
    ["#4F46E5", "#7C3AED"]
):
    words = " ".join(df[df["is_sarcastic"] == label]["clean_headline"]).split()
    top_words = Counter(words).most_common(15)
    word_labels, word_counts = zip(*top_words)
    ax.barh(word_labels[::-1], word_counts[::-1], color=color, alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Frequency")

plt.suptitle("Most Frequent Words by Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("top_words.png", bbox_inches="tight", dpi=150)
plt.show()


## Section 5: Train and Test Split (80/20)

We split our data into a training set and a test set:
- Training set (80%): used to train and tune all models
- Test set (20%): held out until the very end to measure real-world performance

We use stratify=y to make sure both splits have the same proportion of sarcastic vs not sarcastic headlines. This is important so our results are fair and comparable.


In [13]:
X = df["clean_headline"]
y = df["is_sarcastic"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training set size :", len(X_train), "samples -", round(len(X_train)/len(X)*100), "%")
print("Test set size     :", len(X_test),  "samples -", round(len(X_test)/len(X)*100), "%")
print()
print("Class balance in training set:", round(y_train.mean() * 100, 1), "% sarcastic")
print("Class balance in test set     :", round(y_test.mean() * 100, 1), "% sarcastic")


Training set size : 22800 samples - 80 %
Test set size     : 5701 samples - 20 %

Class balance in training set: 47.5 % sarcastic
Class balance in test set     : 47.6 % sarcastic


## Section 6: Classical Machine Learning with TF-IDF

### What is TF-IDF?

TF-IDF stands for Term Frequency - Inverse Document Frequency. It converts text into numbers so machine learning models can process it.

Here is the idea in plain English:
- Term Frequency: how often a word appears in one headline
- Inverse Document Frequency: how rare that word is across all headlines

If a word appears often in one headline but rarely in others, it gets a high score. Common words like "the" appear everywhere and get low scores.

We also use n-grams (combinations of 1, 2, and 3 consecutive words) so the model can recognize phrases like "area man" or "scientists discover" which are strong sarcasm indicators.

### Why start with classical ML?
Classical ML models are fast, interpretable, and often give surprisingly good results. They serve as our baseline before we move to deep learning.


In [14]:
# Create TF-IDF features from the cleaned headlines
tfidf = TfidfVectorizer(
    max_features=20000,  # keep the 20,000 most important terms
    ngram_range=(1, 2),  # use single words and two-word combinations
    sublinear_tf=True    # use log scale for term frequency
)

# Fit on training data only, then transform both sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print("TF-IDF training matrix shape:", X_train_tfidf.shape)
print("TF-IDF test matrix shape     :", X_test_tfidf.shape)
print()
print("Sample features from TF-IDF vocabulary:")
print(tfidf.get_feature_names_out()[:20].tolist())


TF-IDF training matrix shape: (22800, 20000)
TF-IDF test matrix shape     : (5701, 20000)

Sample features from TF-IDF vocabulary:
['aaron', 'aaron hernandez', 'aarp', 'ab', 'abandon', 'abandoned', 'abandoning', 'abbas', 'abbey', 'abby', 'abc', 'abdomen', 'abducted', 'abduction', 'abdul', 'ability', 'able', 'aboard', 'aborted', 'abortion']


In [15]:
# Train four different classical ML models and compare them
models = {
    "Logistic Regression"   : LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(alpha=0.1),
    "Linear SVM"            : LinearSVC(max_iter=3000, random_state=42),
    "Random Forest"         : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

baseline_results = []
fitted_models    = {}

for name, model in models.items():
    print("Training", name, "...")
    model.fit(X_train_tfidf, y_train)
    fitted_models[name] = model

    train_pred = model.predict(X_train_tfidf)
    test_pred  = model.predict(X_test_tfidf)

    baseline_results.append({
        "Model"          : name,
        "Train Accuracy" : round(accuracy_score(y_train, train_pred), 4),
        "Test Accuracy"  : round(accuracy_score(y_test,  test_pred),  4),
        "Test F1 Score"  : round(f1_score(y_test, test_pred), 4),
        "Test Precision" : round(precision_score(y_test, test_pred), 4),
        "Test Recall"    : round(recall_score(y_test, test_pred), 4),
    })

baseline_df = pd.DataFrame(baseline_results).sort_values("Test Accuracy", ascending=False)
print()
print("Baseline Model Comparison:")
baseline_df


Training Logistic Regression ...
Training Multinomial Naive Bayes ...
Training Linear SVM ...
Training Random Forest ...

Baseline Model Comparison:


,Model,Train Accuracy,Test Accuracy,Test F1 Score,Test Precision,Test Recall
2,Linear SVM,0.9704,0.7962,0.7837,0.7911,0.7765
0,Logistic Regression,0.8946,0.7900,0.7729,0.7957,0.7514
1,Multinomial Naive Bayes,0.9245,0.7862,0.7749,0.7759,0.7739
3,Random Forest,0.9997,0.7651,0.7379,0.7861,0.6953


In [16]:
# Visualize the comparison across models
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ["Test Accuracy", "Test F1 Score", "Test Precision"]
colors  = ["#4F46E5", "#7C3AED", "#6D28D9"]

for ax, metric, color in zip(axes, metrics, colors):
    ax.barh(baseline_df["Model"], baseline_df[metric],
            color=color, alpha=0.85, edgecolor="white")
    ax.set_xlim(0.60, 0.90)
    ax.set_title(metric, fontweight="bold")
    ax.set_xlabel("Score")
    for i, v in enumerate(baseline_df[metric]):
        ax.text(v + 0.002, i, str(round(v, 3)), va="center", fontsize=9)

plt.suptitle("Classical Machine Learning - Baseline Model Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("baseline_comparison.png", bbox_inches="tight", dpi=150)
plt.show()


In [17]:
# Print detailed classification report for the best baseline model
best_baseline_name  = baseline_df.iloc[0]["Model"]
best_baseline_model = fitted_models[best_baseline_name]

print("Best baseline model:", best_baseline_name)
print()
print("Training Set Report:")
print(classification_report(y_train, best_baseline_model.predict(X_train_tfidf),
                             target_names=["Not Sarcastic", "Sarcastic"]))
print("Test Set Report:")
print(classification_report(y_test, best_baseline_model.predict(X_test_tfidf),
                             target_names=["Not Sarcastic", "Sarcastic"]))


Best baseline model: Linear SVM

Training Set Report:
               precision    recall  f1-score   support

Not Sarcastic       0.97      0.97      0.97     11959
    Sarcastic       0.97      0.97      0.97     10841

     accuracy                           0.97     22800
    macro avg       0.97      0.97      0.97     22800
 weighted avg       0.97      0.97      0.97     22800

Test Set Report:
               precision    recall  f1-score   support

Not Sarcastic       0.80      0.81      0.81      2990
    Sarcastic       0.79      0.78      0.78      2711

     accuracy                           0.80      5701
    macro avg       0.80      0.80      0.80      5701
 weighted avg       0.80      0.80      0.80      5701



## Section 7: Hyperparameter Tuning and Ensemble Models

The baseline models gave us a good start. Now we push further using two strategies:

### Strategy 1: Hyperparameter Tuning
Every model has settings called hyperparameters that control how it learns. Instead of guessing the best values, we test many combinations automatically using cross-validation and pick the one that performs best. This process is called Grid Search or Random Search.

For example, Logistic Regression has a setting called C that controls regularization (how strongly the model is penalized for being too complex). We test C values of 0.5, 1, 2, 5, and 10.

### Strategy 2: Ensemble Methods
Instead of relying on one model, we combine multiple models:
- Voting Classifier: each model votes on the prediction and the majority wins. With soft voting, we average the probability scores.
- Stacking Classifier: the predictions from multiple base models are fed as inputs to a meta-model (in our case, Logistic Regression) which makes the final decision. This is more powerful than simple voting.


In [18]:
# Use a richer TF-IDF for the tuned models - trigrams and more features
tfidf_v2 = TfidfVectorizer(max_features=20000, ngram_range=(1, 3),
                            sublinear_tf=True, min_df=2)
X_train_tfidf_v2 = tfidf_v2.fit_transform(X_train)
X_test_tfidf_v2  = tfidf_v2.transform(X_test)
print("Enhanced TF-IDF shape:", X_train_tfidf_v2.shape)
print("Now includes unigrams, bigrams, and trigrams for richer features")


Enhanced TF-IDF shape: (22800, 20000)
Now includes unigrams, bigrams, and trigrams for richer features


In [19]:
# Tune Logistic Regression - find the best C value and solver
print("Tuning Logistic Regression...")
lr_params = {
    "C"      : [0.5, 1, 2, 5, 10],
    "penalty": ["l2"],
    "solver" : ["liblinear", "lbfgs"]
}
lr_search = RandomizedSearchCV(
    LogisticRegression(max_iter=2000, random_state=42),
    lr_params, n_iter=8, cv=3, scoring="f1", n_jobs=-1, random_state=42
)
lr_search.fit(X_train_tfidf_v2, y_train)
best_lr = lr_search.best_estimator_

print("Best parameters found:", lr_search.best_params_)
print("Test Accuracy:", round(accuracy_score(y_test, best_lr.predict(X_test_tfidf_v2)), 4))


Tuning Logistic Regression...
Best parameters found: {'solver': 'lbfgs', 'penalty': 'l2', 'C': 5}
Test Accuracy: 0.7962


In [20]:
# Tune Linear SVM - find the best C value
# We also calibrate the SVM so it can output probabilities, which is needed for ensemble
print("Tuning Linear SVM...")
svm_params = {"C": [0.1, 0.5, 1, 2, 5]}
svm_search = RandomizedSearchCV(
    LinearSVC(random_state=42, max_iter=5000),
    svm_params, n_iter=5, cv=3, scoring="f1", n_jobs=-1, random_state=42
)
svm_search.fit(X_train_tfidf_v2, y_train)

best_svm = CalibratedClassifierCV(
    LinearSVC(random_state=42, max_iter=5000, C=svm_search.best_params_["C"]), cv=3
)
best_svm.fit(X_train_tfidf_v2, y_train)

print("Best C value found:", svm_search.best_params_)
print("Test Accuracy:", round(accuracy_score(y_test, best_svm.predict(X_test_tfidf_v2)), 4))


Tuning Linear SVM...
Best C value found: {'C': 0.5}
Test Accuracy: 0.7951


In [21]:
# Tune Naive Bayes - find the best alpha (smoothing) value
print("Tuning Multinomial Naive Bayes...")
nb_params = {"alpha": [0.01, 0.1, 0.5, 1.0, 2.0]}
nb_search = GridSearchCV(MultinomialNB(), nb_params, cv=3, scoring="f1", n_jobs=-1)
nb_search.fit(X_train_tfidf_v2, y_train)
best_nb = nb_search.best_estimator_

print("Best alpha found:", nb_search.best_params_)
print("Test Accuracy:", round(accuracy_score(y_test, best_nb.predict(X_test_tfidf_v2)), 4))


Tuning Multinomial Naive Bayes...
Best alpha found: {'alpha': 0.5}
Test Accuracy: 0.7927


In [22]:
# Build Voting Ensemble - combine all three tuned models
# LR and SVM get weight 2, NB gets weight 1 because LR and SVM perform better
print("Building Voting Ensemble...")
voting_clf = VotingClassifier(
    estimators=[("lr", best_lr), ("svm", best_svm), ("nb", best_nb)],
    voting="soft",
    weights=[2, 2, 1]
)
voting_clf.fit(X_train_tfidf_v2, y_train)
print("Voting Ensemble Test Accuracy:",
      round(accuracy_score(y_test, voting_clf.predict(X_test_tfidf_v2)), 4))


Building Voting Ensemble...
Voting Ensemble Test Accuracy: 0.7995


In [23]:
# Build Stacking Ensemble - use a meta-model to combine predictions
# The three base models make predictions, and Logistic Regression learns how to combine them
print("Building Stacking Ensemble...")
stacking_clf = StackingClassifier(
    estimators=[("lr", best_lr), ("svm", best_svm), ("nb", best_nb)],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=3, n_jobs=-1
)
stacking_clf.fit(X_train_tfidf_v2, y_train)
print("Stacking Ensemble Test Accuracy:",
      round(accuracy_score(y_test, stacking_clf.predict(X_test_tfidf_v2)), 4))


Building Stacking Ensemble...
Stacking Ensemble Test Accuracy: 0.8046


In [24]:
# Compare all tuned and ensemble models together
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    tr_pred = model.predict(X_tr)
    te_pred = model.predict(X_te)
    return {
        "Model"          : name,
        "Train Accuracy" : round(accuracy_score(y_tr, tr_pred), 4),
        "Test Accuracy"  : round(accuracy_score(y_te, te_pred), 4),
        "Test F1 Score"  : round(f1_score(y_te, te_pred), 4),
        "Test Precision" : round(precision_score(y_te, te_pred), 4),
        "Test Recall"    : round(recall_score(y_te, te_pred), 4),
    }

tuned_results = [
    evaluate_model("Logistic Regression (Tuned)", best_lr,      X_train_tfidf_v2, y_train, X_test_tfidf_v2, y_test),
    evaluate_model("SVM (Tuned)",                 best_svm,     X_train_tfidf_v2, y_train, X_test_tfidf_v2, y_test),
    evaluate_model("Naive Bayes (Tuned)",         best_nb,      X_train_tfidf_v2, y_train, X_test_tfidf_v2, y_test),
    evaluate_model("Voting Ensemble",             voting_clf,   X_train_tfidf_v2, y_train, X_test_tfidf_v2, y_test),
    evaluate_model("Stacking Ensemble - BEST",   stacking_clf, X_train_tfidf_v2, y_train, X_test_tfidf_v2, y_test),
]
tuned_df = pd.DataFrame(tuned_results).sort_values("Test F1 Score", ascending=False)
print("Tuned and Ensemble Model Comparison:")
tuned_df


Tuned and Ensemble Model Comparison:


,Model,Train Accuracy,Test Accuracy,Test F1 Score,Test Precision,Test Recall
4,Stacking Ensemble - BEST,0.9293,0.8046,0.7939,0.7963,0.7916
3,Voting Ensemble,0.9386,0.7995,0.7869,0.7956,0.7783
0,Logistic Regression (Tuned),0.9478,0.7962,0.7829,0.7933,0.7728
1,SVM (Tuned),0.9330,0.7951,0.7827,0.7897,0.7757
2,Naive Bayes (Tuned),0.9104,0.7927,0.7789,0.7901,0.7680


In [25]:
# Final evaluation of the best classical model - Stacking Ensemble
y_pred_stacking  = stacking_clf.predict(X_test_tfidf_v2)
y_proba_stacking = stacking_clf.predict_proba(X_test_tfidf_v2)[:, 1]
auc_stacking     = roc_auc_score(y_test, y_proba_stacking)

print("Stacking Ensemble Final Results:")
print("Test Accuracy  :", round(accuracy_score(y_test, y_pred_stacking), 4))
print("Test F1 Score  :", round(f1_score(y_test, y_pred_stacking), 4))
print("Test ROC-AUC   :", round(auc_stacking, 4))
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix shows which class the model gets right or wrong
cm = confusion_matrix(y_test, y_pred_stacking)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Not Sarcastic", "Sarcastic"],
            yticklabels=["Not Sarcastic", "Sarcastic"])
axes[0].set_title("Confusion Matrix - Stacking Ensemble", fontweight="bold")
axes[0].set_ylabel("Actual Label")
axes[0].set_xlabel("Predicted Label")

# ROC Curve shows how well the model separates the two classes
fpr, tpr, _ = roc_curve(y_test, y_proba_stacking)
axes[1].plot(fpr, tpr, color="#7C3AED", lw=2, label="AUC = " + str(round(auc_stacking, 3)))
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="Random Classifier")
axes[1].set_title("ROC Curve - Stacking Ensemble", fontweight="bold")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend()

plt.tight_layout()
plt.savefig("stacking_ensemble_results.png", bbox_inches="tight", dpi=150)
plt.show()


Stacking Ensemble Final Results:
Test Accuracy  : 0.8046
Test F1 Score  : 0.7939
Test ROC-AUC   : 0.8885



In [26]:
# Save the classical ML models so they can be used in the Streamlit app
os.makedirs("models", exist_ok=True)

with open("models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_v2, f)

with open("models/sarcasm_model.pkl", "wb") as f:
    pickle.dump(stacking_clf, f)

metadata_ml = {
    "model_name"    : "Stacking Ensemble (LR + SVM + NB -> Logistic Regression)",
    "test_accuracy" : round(accuracy_score(y_test, y_pred_stacking), 4),
    "test_f1"       : round(f1_score(y_test, y_pred_stacking), 4),
    "test_auc"      : round(auc_stacking, 4),
    "n_features"    : X_train_tfidf_v2.shape[1],
}
with open("models/model_metadata.pkl", "wb") as f:
    pickle.dump(metadata_ml, f)

print("Classical ML models saved successfully!")
print(metadata_ml)


Classical ML models saved successfully!
{'model_name': 'Stacking Ensemble (LR + SVM + NB -> Logistic Regression)', 'test_accuracy': 0.8046, 'test_f1': 0.7939, 'test_auc': 0.8885, 'n_features': 20000}


## Section 8: Deep Learning - RNN, GRU, LSTM, and BiLSTM

### Why move to Deep Learning?

TF-IDF treats every word independently. It has no concept of word order or context. The sentence "I am not happy" looks the same to TF-IDF whether "not" comes before or after "happy".

Deep Learning models, especially recurrent networks, read text one word at a time and remember what came before. This is much more natural and powerful for understanding language.

### How text is prepared for Deep Learning

Instead of TF-IDF, we use a different approach:
- Tokenizer: assigns a unique number to each word. For example, "area" might become 142, "man" might become 88
- Padding: all headlines are made the same length by adding zeros at the end
- Embedding: each word number gets converted into a list of numbers (a vector) that the model learns during training

### The models we will train

1. Simple RNN: the most basic recurrent model. Reads words one by one and maintains a memory state. Tends to forget information from early in the sequence.

2. GRU (Gated Recurrent Unit): an improved version of RNN with gates that control what information to remember and forget.

3. LSTM (Long Short-Term Memory): similar to GRU but with separate memory cells. Very good at remembering long-range patterns.

4. Bidirectional LSTM: reads the headline both forward (left to right) and backward (right to left). Sarcasm often comes at the end of a sentence, so reading backwards helps catch it.


In [27]:
# Settings for the deep learning models
MAX_VOCAB = 20000   # how many unique words to keep in vocabulary
MAX_LEN   = 30      # maximum headline length in words (pad shorter, truncate longer)
EMBED_DIM = 128     # size of the word embedding vectors

# The tokenizer learns which words exist and assigns each one a number
tokenizer_dl = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer_dl.fit_on_texts(X_train)

# Convert each headline from words to a list of numbers
X_train_seq = tokenizer_dl.texts_to_sequences(X_train)
X_test_seq  = tokenizer_dl.texts_to_sequences(X_test)

# Make all sequences the same length by padding with zeros
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding="post", truncating="post")

y_train_arr = np.array(y_train)
y_test_arr  = np.array(y_test)

print("Vocabulary size   :", len(tokenizer_dl.word_index))
print("Training shape    :", X_train_pad.shape)
print("Test shape        :", X_test_pad.shape)
print()
print("Example transformation:")
print("Original headline :", X_train.iloc[0])
print("As word numbers   :", X_train_seq[0][:10], "...")
print("After padding     :", X_train_pad[0])


Vocabulary size   : 20525
Training shape    : (22800, 30)
Test shape        : (5701, 30)

Example transformation:
Original headline : trump lawyer start couch statement russia investigation
As word numbers   : [2, 648, 295, 2119, 1327, 395, 489] ...
After padding     : [   2  648  295 2119 1327  395  489    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]


In [28]:
# Helper functions used by all deep learning models
dl_results = []

def get_callbacks():
    # Early stopping stops training when the model stops improving on validation data
    # ReduceLROnPlateau reduces the learning rate when training gets stuck
    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6)
    ]

def evaluate_dl_model(name, model, X_tr, y_tr, X_te, y_te):
    y_pred  = (model.predict(X_te, verbose=0) > 0.5).astype(int).flatten()
    y_proba = model.predict(X_te, verbose=0).flatten()
    row = {
        "Model"          : name,
        "Train Accuracy" : round(accuracy_score(y_tr, (model.predict(X_tr, verbose=0) > 0.5).astype(int).flatten()), 4),
        "Test Accuracy"  : round(accuracy_score(y_te, y_pred), 4),
        "Test F1 Score"  : round(f1_score(y_te, y_pred), 4),
        "Test AUC"       : round(roc_auc_score(y_te, y_proba), 4),
    }
    dl_results.append(row)
    print(name, "-> Test Accuracy:", row["Test Accuracy"],
          "| F1:", row["Test F1 Score"], "| AUC:", row["Test AUC"])
    return row

def plot_training_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["accuracy"],     label="Training",   color="#4F46E5")
    axes[0].plot(history.history["val_accuracy"], label="Validation", color="#7C3AED")
    axes[0].set_title(title + " - Accuracy per Epoch")
    axes[0].legend()
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")

    axes[1].plot(history.history["loss"],     label="Training",   color="#4F46E5")
    axes[1].plot(history.history["val_loss"], label="Validation", color="#7C3AED")
    axes[1].set_title(title + " - Loss per Epoch")
    axes[1].legend()
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")

    plt.tight_layout()
    plt.show()

print("Helper functions defined and ready to use.")


Helper functions defined and ready to use.


In [29]:
# Model 1: Simple RNN
# This is the most basic recurrent model. It often struggles with long sequences
# because it tends to forget information from the beginning of the headline.
print("Training Simple RNN...")

rnn_model = Sequential([
    Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    SimpleRNN(64, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(0.01)),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])
rnn_model.compile(
    optimizer=Adam(learning_rate=0.0005, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
rnn_model.build(input_shape=(None, MAX_LEN))
rnn_model.summary()

history_rnn = rnn_model.fit(
    X_train_pad, y_train_arr,
    epochs=20, batch_size=64,
    validation_split=0.1,
    callbacks=get_callbacks(),
    verbose=1
)

plot_training_history(history_rnn, "Simple RNN")
evaluate_dl_model("Simple RNN", rnn_model, X_train_pad, y_train_arr, X_test_pad, y_test_arr)


Training Simple RNN...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 30, 128)        │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 30, 128)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,574,465 (9.82 MB)

 Trainable params: 2,574,465 (9.82 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
321/321 ━━━━━━━━━━━━━━━━━━━━ 29s 43ms/step - accuracy: 0.5053 - loss: 1.1391 - val_accuracy: 0.5224 - val_loss: 0.8595 - learning_rate: 5.0000e-04
Epoch 2/20
321/321 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.5180 - loss: 0.7763 - val_accuracy: 0.5224 - val_loss: 0.7247 - learning_rate: 5.0000e-04
Epoch 3/20
321/321 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.5192 - loss: 0.7118 - val_accuracy: 0.5224 - val_loss: 0.7015 - learning_rate: 5.0000e-04
Epoch 4/20
321/321 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.5297 - loss: 0.6999 - val_accuracy: 0.5224 - val_loss: 0.6963 - learning_rate: 5.0000e-04
Epoch 5/20
321/321 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.5625 - loss: 0.6899 - val_accuracy: 0.6544 - val_loss: 0.6374 - learning_rate: 5.0000e-04
Epoch 6/20
321/321 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.6811 - loss: 0.6279 - val_accuracy: 0.7189 - val_loss: 0.5826 - learning_rate: 5.0000e-04
Epoch 7/20
321/321 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/ste

{'Model': 'Simple RNN',
 'Train Accuracy': 0.8081,
 'Test Accuracy': 0.7215,
 'Test F1 Score': 0.6825,
 'Test AUC': 0.7763}

In [30]:
# Model 2: GRU (Gated Recurrent Unit)
# GRU improves on simple RNN by using gates that decide what to remember and forget.
# It is faster to train than LSTM and often performs similarly.
print("Training GRU...")

gru_model = Sequential([
    Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    GRU(128, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(0.01)),
    Dense(64, activation="relu", kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])
gru_model.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
gru_model.build(input_shape=(None, MAX_LEN))

history_gru = gru_model.fit(
    X_train_pad, y_train_arr,
    epochs=15, batch_size=128,
    validation_split=0.1,
    callbacks=get_callbacks(),
    verbose=1
)

plot_training_history(history_gru, "GRU")
evaluate_dl_model("GRU", gru_model, X_train_pad, y_train_arr, X_test_pad, y_test_arr)


Training GRU...
Epoch 1/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 3886s 24s/step - accuracy: 0.5234 - loss: 1.3347 - val_accuracy: 0.5224 - val_loss: 0.7244 - learning_rate: 0.0010
Epoch 2/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 69s 427ms/step - accuracy: 0.5248 - loss: 0.6993 - val_accuracy: 0.5224 - val_loss: 0.6925 - learning_rate: 0.0010
Epoch 3/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 65s 397ms/step - accuracy: 0.5248 - loss: 0.6920 - val_accuracy: 0.5224 - val_loss: 0.6923 - learning_rate: 0.0010
Epoch 4/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 57s 353ms/step - accuracy: 0.5248 - loss: 0.6920 - val_accuracy: 0.5224 - val_loss: 0.6922 - learning_rate: 0.0010
Epoch 5/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 57s 351ms/step - accuracy: 0.5248 - loss: 0.6920 - val_accuracy: 0.5224 - val_loss: 0.6923 - learning_rate: 0.0010
Epoch 6/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 65s 401ms/step - accuracy: 0.5248 - loss: 0.6919 - val_accuracy: 0.5224 - val_loss: 0.6922 - learning_rate: 0.0010
Epoch 7/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 57s 350ms/step

{'Model': 'GRU',
 'Train Accuracy': 0.5245,
 'Test Accuracy': 0.5245,
 'Test F1 Score': 0.0,
 'Test AUC': 0.5002}

In [31]:
# Model 3: LSTM (Long Short-Term Memory)
# LSTM is designed to remember information over long sequences.
# It has separate input, forget, and output gates which give it better memory.
print("Training LSTM...")

lstm_model = Sequential([
    Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(0.01)),
    Dense(64, activation="relu", kernel_regularizer=l2(0.01)),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])
lstm_model.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
lstm_model.build(input_shape=(None, MAX_LEN))

history_lstm = lstm_model.fit(
    X_train_pad, y_train_arr,
    epochs=15, batch_size=128,
    validation_split=0.1,
    callbacks=get_callbacks(),
    verbose=1
)

plot_training_history(history_lstm, "LSTM")
evaluate_dl_model("LSTM", lstm_model, X_train_pad, y_train_arr, X_test_pad, y_test_arr)


Training LSTM...
Epoch 1/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 100s 357ms/step - accuracy: 0.5240 - loss: 1.3122 - val_accuracy: 0.5224 - val_loss: 0.7224 - learning_rate: 0.0010
Epoch 2/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 51s 313ms/step - accuracy: 0.5248 - loss: 0.6991 - val_accuracy: 0.5224 - val_loss: 0.6924 - learning_rate: 0.0010
Epoch 3/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 49s 304ms/step - accuracy: 0.5248 - loss: 0.6921 - val_accuracy: 0.5224 - val_loss: 0.6922 - learning_rate: 0.0010
Epoch 4/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 50s 308ms/step - accuracy: 0.5248 - loss: 0.6920 - val_accuracy: 0.5224 - val_loss: 0.6922 - learning_rate: 0.0010
Epoch 5/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 48s 297ms/step - accuracy: 0.5248 - loss: 0.6920 - val_accuracy: 0.5224 - val_loss: 0.6922 - learning_rate: 0.0010
Epoch 6/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 48s 297ms/step - accuracy: 0.5248 - loss: 0.6920 - val_accuracy: 0.5224 - val_loss: 0.6922 - learning_rate: 0.0010
Epoch 7/15
161/161 ━━━━━━━━━━━━━━━━━━━━ 50s 309ms/st

{'Model': 'LSTM',
 'Train Accuracy': 0.5245,
 'Test Accuracy': 0.5245,
 'Test F1 Score': 0.0,
 'Test AUC': 0.5}

In [32]:
# Model 4: Bidirectional LSTM
# This model reads the headline in both directions - left to right AND right to left.
# Sarcasm often comes at the end of a sentence, so reading backwards helps the model
# connect the punchline to the setup at the beginning.
print("Training Bidirectional LSTM...")

bilstm_model = Sequential([
    Embedding(MAX_VOCAB, EMBED_DIM, input_length=MAX_LEN),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2, kernel_regularizer=l2(0.01))),
    BatchNormalization(),
    Dense(64, activation="relu", kernel_regularizer=l2(0.01)),
    Dropout(0.4),
    Dense(1, activation="sigmoid")
])
bilstm_model.compile(
    optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
bilstm_model.build(input_shape=(None, MAX_LEN))

history_bilstm = bilstm_model.fit(
    X_train_pad, y_train_arr,
    epochs=20, batch_size=128,
    validation_split=0.1,
    callbacks=get_callbacks(),
    verbose=1
)

plot_training_history(history_bilstm, "Bidirectional LSTM")
evaluate_dl_model("Bidirectional LSTM", bilstm_model, X_train_pad, y_train_arr, X_test_pad, y_test_arr)


Training Bidirectional LSTM...
Epoch 1/20
161/161 ━━━━━━━━━━━━━━━━━━━━ 145s 662ms/step - accuracy: 0.7221 - loss: 1.7860 - val_accuracy: 0.7294 - val_loss: 0.7659 - learning_rate: 0.0010
Epoch 2/20
161/161 ━━━━━━━━━━━━━━━━━━━━ 92s 568ms/step - accuracy: 0.8672 - loss: 0.3733 - val_accuracy: 0.8004 - val_loss: 0.5681 - learning_rate: 0.0010
Epoch 3/20
161/161 ━━━━━━━━━━━━━━━━━━━━ 89s 550ms/step - accuracy: 0.9201 - loss: 0.2357 - val_accuracy: 0.7917 - val_loss: 0.4858 - learning_rate: 0.0010
Epoch 4/20
161/161 ━━━━━━━━━━━━━━━━━━━━ 90s 560ms/step - accuracy: 0.9458 - loss: 0.1770 - val_accuracy: 0.7803 - val_loss: 0.5091 - learning_rate: 0.0010
Epoch 5/20
161/161 ━━━━━━━━━━━━━━━━━━━━ 88s 544ms/step - accuracy: 0.9622 - loss: 0.1347 - val_accuracy: 0.7811 - val_loss: 0.6705 - learning_rate: 0.0010
Epoch 6/20
161/161 ━━━━━━━━━━━━━━━━━━━━ 87s 542ms/step - accuracy: 0.9689 - loss: 0.1155 - val_accuracy: 0.7715 - val_loss: 0.7473 - learning_rate: 0.0010
Epoch 7/20
161/161 ━━━━━━━━━━━━━━━━━━━

{'Model': 'Bidirectional LSTM',
 'Train Accuracy': 0.9461,
 'Test Accuracy': 0.7928,
 'Test F1 Score': 0.7646,
 'Test AUC': 0.881}

In [33]:
# Compare all deep learning models
dl_df = pd.DataFrame(dl_results).sort_values("Test F1 Score", ascending=False)
print("Deep Learning Model Comparison:")
print(dl_df.to_string(index=False))
dl_df


Deep Learning Model Comparison:
             Model  Train Accuracy  Test Accuracy  Test F1 Score  Test AUC
Bidirectional LSTM          0.9461         0.7928         0.7646    0.8810
        Simple RNN          0.8081         0.7215         0.6825    0.7763
               GRU          0.5245         0.5245         0.0000    0.5002
              LSTM          0.5245         0.5245         0.0000    0.5000


,Model,Train Accuracy,Test Accuracy,Test F1 Score,Test AUC
3,Bidirectional LSTM,0.9461,0.7928,0.7646,0.8810
0,Simple RNN,0.8081,0.7215,0.6825,0.7763
1,GRU,0.5245,0.5245,0.0000,0.5002
2,LSTM,0.5245,0.5245,0.0000,0.5000


## Section 9: GloVe Pre-trained Embeddings - Our Best Model

### Why are we still not above 85%?

With random embeddings, our deep learning models must learn the meaning of every word from scratch using only 28,000 news headlines. That is simply not enough data to learn rich word representations.

### The Solution: GloVe

GloVe stands for Global Vectors for Word Representation. It was developed by Stanford and was trained on 6 billion words from Wikipedia and news articles.

Instead of starting with random numbers for each word, we start with vectors that already encode meaning. GloVe already knows that:
- "scientist" and "discover" appear together in research contexts
- "water is wet" is a trivially obvious statement (sarcasm signal)
- "area man" is typical of satirical news writing

This is the difference between learning from 28,000 examples versus starting with knowledge from 6 billion sentences.

### Key differences in preprocessing for GloVe

In our classical ML and basic DL sections, we removed stopwords. For GloVe, we keep them. Phrases like "is, in fact", "what a surprise", and "yet another" carry strong sarcasm signals that would be lost if we removed the connecting words.

We also use a longer sequence length of 40 words instead of 30, to capture the full headline context.


In [34]:
# Download GloVe word vectors (this is a one-time download of about 822 MB)
GLOVE_DIR  = "data/glove"
GLOVE_FILE = os.path.join(GLOVE_DIR, "glove.6B.100d.txt")
GLOVE_ZIP  = os.path.join(GLOVE_DIR, "glove.6B.zip")
os.makedirs(GLOVE_DIR, exist_ok=True)

if not os.path.exists(GLOVE_FILE):
    print("Downloading GloVe vectors. This may take a few minutes...")
    url = "https://nlp.stanford.edu/data/glove.6B.zip"
    urllib.request.urlretrieve(url, GLOVE_ZIP)
    print("Extracting files...")
    with zipfile.ZipFile(GLOVE_ZIP) as z:
        z.extract("glove.6B.100d.txt", GLOVE_DIR)
    os.remove(GLOVE_ZIP)
    print("GloVe download complete!")
else:
    print("GloVe vectors already downloaded. Skipping download.")


GloVe vectors already downloaded. Skipping download.


In [35]:
# Load the GloVe vectors into memory
# Each line in the file has a word followed by 100 numbers that represent its meaning
print("Loading GloVe vectors into memory...")
embeddings_index = {}
with open(GLOVE_FILE, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word   = values[0]
        vector = np.array(values[1:], dtype="float32")
        embeddings_index[word] = vector

print("Total word vectors loaded:", len(embeddings_index))
print("Each vector has", len(embeddings_index["the"]), "dimensions")
print()
print("Example - first 5 values of the vector for the word 'sarcasm':")
if "sarcasm" in embeddings_index:
    print(embeddings_index["sarcasm"][:5])
else:
    print("Word not in GloVe vocabulary")


Loading GloVe vectors into memory...
Total word vectors loaded: 400000
Each vector has 100 dimensions

Example - first 5 values of the vector for the word 'sarcasm':
[-0.0086549 -0.021338   0.92552   -0.55249   -0.088089 ]


In [36]:
# GloVe preprocessing - lighter cleaning that keeps stopwords
# Important: we fit the tokenizer on the FULL dataset before splitting.
# This gives richer vocabulary coverage and is exactly what the original notebook did.
def clean_text_glove(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

MAX_VOCAB_G = 20000
MAX_LEN_G   = 40
EMBED_DIM_G = 100

# Apply light cleaning to full dataset
df["glove_headline"] = df["headline"].apply(clean_text_glove)
df = df[df["glove_headline"].str.len() > 0].reset_index(drop=True)

# Fit tokenizer on the FULL dataset (not just training) - same as original notebook
tokenizer_g = Tokenizer(num_words=MAX_VOCAB_G, oov_token="<OOV>")
tokenizer_g.fit_on_texts(df["glove_headline"])

# Now create sequences from train and test sets
X_train_g = df.loc[X_train.index, "glove_headline"] if hasattr(X_train, 'index') else X_train.map(clean_text_glove)
X_test_g  = df.loc[X_test.index,  "glove_headline"] if hasattr(X_test,  'index') else X_test.map(clean_text_glove)

X_train_seq_g = tokenizer_g.texts_to_sequences(X_train_g)
X_test_seq_g  = tokenizer_g.texts_to_sequences(X_test_g)

X_train_pad_g = pad_sequences(X_train_seq_g, maxlen=MAX_LEN_G, padding="post", truncating="post")
X_test_pad_g  = pad_sequences(X_test_seq_g,  maxlen=MAX_LEN_G, padding="post", truncating="post")

print("GloVe training shape:", X_train_pad_g.shape)
print("GloVe test shape     :", X_test_pad_g.shape)
print("Vocabulary size      :", len(tokenizer_g.word_index))


GloVe training shape: (22800, 40)
GloVe test shape     : (5701, 40)
Vocabulary size      : 25915


In [37]:
# Build embedding matrix aligned with our tokenizer vocabulary
word_index       = tokenizer_g.word_index
num_words_g      = min(MAX_VOCAB_G, len(word_index) + 1)
embedding_matrix = np.zeros((num_words_g, EMBED_DIM_G))
words_found      = 0

for word, idx in word_index.items():
    if idx < num_words_g:
        vec = embeddings_index.get(word)
        if vec is not None:
            embedding_matrix[idx] = vec
            words_found += 1

print("Words in tokenizer      :", len(word_index))
print("Words found in GloVe    :", words_found, "-", round(words_found/len(word_index)*100, 1), "%")
print("Embedding matrix shape  :", embedding_matrix.shape)

# Embedding layer factory function - same as original notebook
def create_glove_embedding(trainable=False):
    return Embedding(
        input_dim=num_words_g,
        output_dim=EMBED_DIM_G,
        weights=[embedding_matrix],
        input_length=MAX_LEN_G,
        trainable=trainable
    )


Words in tokenizer      : 25915
Words found in GloVe    : 19484 - 75.2 %
Embedding matrix shape  : (20000, 100)


In [38]:
# GloVe Model 1: Bidirectional GRU with frozen GloVe embeddings
print("Training GloVe + Bidirectional GRU...")

bigru_model = Sequential([
    create_glove_embedding(trainable=False),
    SpatialDropout1D(0.3),
    Bidirectional(GRU(128, dropout=0.3, recurrent_dropout=0.2,
                      kernel_regularizer=l2(0.001))),
    BatchNormalization(),
    Dense(64, activation="relu", kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(1, activation="sigmoid")
])
bigru_model.compile(optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
                    loss="binary_crossentropy", metrics=["accuracy"])

history_bigru = bigru_model.fit(
    X_train_pad_g, y_train_arr, epochs=30, batch_size=64,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
               ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6)],
    verbose=1
)
plot_training_history(history_bigru, "GloVe + BiGRU")

y_pred_bigru  = (bigru_model.predict(X_test_pad_g, verbose=0) > 0.5).astype(int).flatten()
y_proba_bigru = bigru_model.predict(X_test_pad_g, verbose=0).flatten()
print("GloVe + BiGRU -> Test Acc:", round(accuracy_score(y_test_arr, y_pred_bigru), 4),
      "F1:", round(f1_score(y_test_arr, y_pred_bigru), 4))


Training GloVe + Bidirectional GRU...
Epoch 1/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 233s 484ms/step - accuracy: 0.6504 - loss: 0.8101 - val_accuracy: 0.7658 - val_loss: 0.6354 - learning_rate: 0.0010
Epoch 2/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 138s 428ms/step - accuracy: 0.7150 - loss: 0.6581 - val_accuracy: 0.7899 - val_loss: 0.5551 - learning_rate: 0.0010
Epoch 3/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 146s 454ms/step - accuracy: 0.7496 - loss: 0.5976 - val_accuracy: 0.7908 - val_loss: 0.5306 - learning_rate: 0.0010
Epoch 4/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 144s 449ms/step - accuracy: 0.7646 - loss: 0.5645 - val_accuracy: 0.8219 - val_loss: 0.4896 - learning_rate: 0.0010
Epoch 5/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 145s 450ms/step - accuracy: 0.7774 - loss: 0.5377 - val_accuracy: 0.8215 - val_loss: 0.4646 - learning_rate: 0.0010
Epoch 6/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 177s 371ms/step - accuracy: 0.7869 - loss: 0.5159 - val_accuracy: 0.8206 - val_loss: 0.4575 - learning_rate: 0.0010
Epoch 7/30
321/321 ━━━━━━━

In [39]:
# GloVe Model 2: Bidirectional LSTM with frozen embeddings
print("Training GloVe + Bidirectional LSTM (Frozen)...")

bilstm_frozen = Sequential([
    create_glove_embedding(trainable=False),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.2,
                       kernel_regularizer=l2(0.001))),
    BatchNormalization(),
    Dense(64, activation="relu", kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(1, activation="sigmoid")
])
bilstm_frozen.compile(optimizer=Adam(learning_rate=0.001, clipnorm=1.0),
                      loss="binary_crossentropy", metrics=["accuracy"])

history_bilstm_frozen = bilstm_frozen.fit(
    X_train_pad_g, y_train_arr, epochs=30, batch_size=64,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
               ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6)],
    verbose=1
)
plot_training_history(history_bilstm_frozen, "GloVe + BiLSTM Frozen")

y_pred_frozen  = (bilstm_frozen.predict(X_test_pad_g, verbose=0) > 0.5).astype(int).flatten()
y_proba_frozen = bilstm_frozen.predict(X_test_pad_g, verbose=0).flatten()
print("GloVe + BiLSTM (Frozen) -> Test Acc:", round(accuracy_score(y_test_arr, y_pred_frozen), 4),
      "F1:", round(f1_score(y_test_arr, y_pred_frozen), 4))


Training GloVe + Bidirectional LSTM (Frozen)...
Epoch 1/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 147s 277ms/step - accuracy: 0.6635 - loss: 0.8687 - val_accuracy: 0.7579 - val_loss: 0.7183 - learning_rate: 0.0010
Epoch 2/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 85s 263ms/step - accuracy: 0.7339 - loss: 0.6731 - val_accuracy: 0.7947 - val_loss: 0.5954 - learning_rate: 0.0010
Epoch 3/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 93s 291ms/step - accuracy: 0.7598 - loss: 0.6023 - val_accuracy: 0.7996 - val_loss: 0.5274 - learning_rate: 0.0010
Epoch 4/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 84s 261ms/step - accuracy: 0.7746 - loss: 0.5596 - val_accuracy: 0.8167 - val_loss: 0.4852 - learning_rate: 0.0010
Epoch 5/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 89s 278ms/step - accuracy: 0.7889 - loss: 0.5304 - val_accuracy: 0.8351 - val_loss: 0.4586 - learning_rate: 0.0010
Epoch 6/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 87s 270ms/step - accuracy: 0.7984 - loss: 0.5057 - val_accuracy: 0.8211 - val_loss: 0.4576 - learning_rate: 0.0010
Epoch 7/30
321/321 ━━

In [40]:
# GloVe Model 3: Bidirectional LSTM with fine-tuned embeddings
print("Training GloVe + Bidirectional LSTM (Fine-tuned)...")

bilstm_tuned = Sequential([
    create_glove_embedding(trainable=True),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.2,
                       kernel_regularizer=l2(0.001))),
    BatchNormalization(),
    Dense(64, activation="relu", kernel_regularizer=l2(0.001)),
    Dropout(0.4),
    Dense(1, activation="sigmoid")
])
bilstm_tuned.compile(optimizer=Adam(learning_rate=0.0005, clipnorm=1.0),
                     loss="binary_crossentropy", metrics=["accuracy"])

history_bilstm_tuned = bilstm_tuned.fit(
    X_train_pad_g, y_train_arr, epochs=30, batch_size=64,
    validation_split=0.1,
    callbacks=[EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
               ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6)],
    verbose=1
)
plot_training_history(history_bilstm_tuned, "GloVe + BiLSTM Fine-tuned")

y_pred_tuned  = (bilstm_tuned.predict(X_test_pad_g, verbose=0) > 0.5).astype(int).flatten()
y_proba_tuned = bilstm_tuned.predict(X_test_pad_g, verbose=0).flatten()
print("GloVe + BiLSTM (Fine-tuned) -> Test Acc:", round(accuracy_score(y_test_arr, y_pred_tuned), 4),
      "F1:", round(f1_score(y_test_arr, y_pred_tuned), 4))


Training GloVe + Bidirectional LSTM (Fine-tuned)...
Epoch 1/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 82s 229ms/step - accuracy: 0.6519 - loss: 0.9180 - val_accuracy: 0.7825 - val_loss: 0.7199 - learning_rate: 5.0000e-04
Epoch 2/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 76s 238ms/step - accuracy: 0.7640 - loss: 0.6829 - val_accuracy: 0.8171 - val_loss: 0.5742 - learning_rate: 5.0000e-04
Epoch 3/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 80s 250ms/step - accuracy: 0.7999 - loss: 0.5785 - val_accuracy: 0.8294 - val_loss: 0.5022 - learning_rate: 5.0000e-04
Epoch 4/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 84s 260ms/step - accuracy: 0.8260 - loss: 0.5062 - val_accuracy: 0.8518 - val_loss: 0.4477 - learning_rate: 5.0000e-04
Epoch 5/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 89s 276ms/step - accuracy: 0.8499 - loss: 0.4463 - val_accuracy: 0.8518 - val_loss: 0.4294 - learning_rate: 5.0000e-04
Epoch 6/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 79s 247ms/step - accuracy: 0.8608 - loss: 0.4071 - val_accuracy: 0.8732 - val_loss: 0.3943 - learning_rate: 5.000

In [41]:
# GloVe Model 4: Stacked Bidirectional LSTM - Our Best Model
# This is copied exactly from the original notebook that achieved 88.9%.
# Three stacked BiLSTM layers: 256 -> 128 -> 64 units.
# Key settings: val_accuracy monitoring, l2=1e-4, batch_size=64
from tensorflow.keras import regularizers

glove_results = []

def get_glove_callbacks():
    # Original notebook monitored val_accuracy not val_loss - this is critical
    return [
        EarlyStopping(monitor="val_accuracy", patience=6,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3,
                          patience=2, min_lr=1e-6, verbose=1),
    ]

stacked_bilstm = Sequential([
    create_glove_embedding(trainable=True),
    SpatialDropout1D(0.3),
    Bidirectional(LSTM(256, dropout=0.3, recurrent_dropout=0.2,
                       return_sequences=True,
                       kernel_regularizer=regularizers.l2(1e-4))),
    Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.2,
                       return_sequences=True,
                       kernel_regularizer=regularizers.l2(1e-4))),
    Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2,
                       kernel_regularizer=regularizers.l2(1e-4))),
    BatchNormalization(),
    Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
], name="GloVe_Stacked_BiLSTM")

stacked_bilstm.build(input_shape=(None, MAX_LEN_G))
stacked_bilstm.compile(
    optimizer=Adam(learning_rate=0.0005, clipnorm=1.0),
    loss="binary_crossentropy", metrics=["accuracy"]
)
stacked_bilstm.summary()

history_stacked = stacked_bilstm.fit(
    X_train_pad_g, y_train_arr,
    epochs=30, batch_size=64,
    validation_split=0.1,
    callbacks=get_glove_callbacks(),
    verbose=1
)

plot_training_history(history_stacked, "GloVe + Stacked BiLSTM")

y_pred_stacked  = (stacked_bilstm.predict(X_test_pad_g, verbose=0) > 0.5).astype(int).flatten()
y_proba_stacked = stacked_bilstm.predict(X_test_pad_g, verbose=0).flatten()

glove_results.append({
    "Model"         : "GloVe + Stacked BiLSTM - BEST",
    "Test Accuracy" : round(accuracy_score(y_test_arr, y_pred_stacked), 4),
    "Test F1 Score" : round(f1_score(y_test_arr, y_pred_stacked), 4),
    "Test AUC"      : round(roc_auc_score(y_test_arr, y_proba_stacked), 4),
})
print("Result:", glove_results[-1])


Model: "GloVe_Stacked_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 40, 100)        │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_7             │ (None, 40, 100)        │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 40, 512)        │       731,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 40, 256)        │       656,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_6 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,577,217 (13.65 MB)

 Trainable params: 3,576,961 (13.65 MB)

 Non-trainable params: 256 (1.00 KB)

Epoch 1/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 1791s 6s/step - accuracy: 0.6192 - loss: 0.8418 - val_accuracy: 0.7487 - val_loss: 0.7063 - learning_rate: 5.0000e-04
Epoch 2/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 395s 1s/step - accuracy: 0.7609 - loss: 0.6533 - val_accuracy: 0.8079 - val_loss: 0.5746 - learning_rate: 5.0000e-04
Epoch 3/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 411s 1s/step - accuracy: 0.8075 - loss: 0.5525 - val_accuracy: 0.8333 - val_loss: 0.4946 - learning_rate: 5.0000e-04
Epoch 4/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 441s 1s/step - accuracy: 0.8383 - loss: 0.4776 - val_accuracy: 0.8434 - val_loss: 0.4536 - learning_rate: 5.0000e-04
Epoch 5/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 427s 1s/step - accuracy: 0.8535 - loss: 0.4344 - val_accuracy: 0.8561 - val_loss: 0.4366 - learning_rate: 5.0000e-04
Epoch 6/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 470s 1s/step - accuracy: 0.8707 - loss: 0.3934 - val_accuracy: 0.8618 - val_loss: 0.4032 - learning_rate: 5.0000e-04
Epoch 7/30
321/321 ━━━━━━━━━━━━━━━━━━━━ 489s 2s/step - ac

In [42]:
# Compare all GloVe models
glove_df = pd.DataFrame(glove_results).sort_values("Test F1 Score", ascending=False)
print("GloVe Model Comparison:")
print(glove_df.to_string(index=False))
glove_df


GloVe Model Comparison:
                        Model  Test Accuracy  Test F1 Score  Test AUC
GloVe + Stacked BiLSTM - BEST         0.8918         0.8859    0.9545


,Model,Test Accuracy,Test F1 Score,Test AUC
0,GloVe + Stacked BiLSTM - BEST,0.8918,0.8859,0.9545


## Section 10: Full Accuracy Journey - All Models Compared

Let us now put everything together and see how accuracy improved as we moved from simple models to more powerful ones.


In [43]:
# Full accuracy journey from baseline to best model
all_results = [
    {"Stage": "Baseline",                "Model": "TF-IDF + Logistic Regression",  "Test Accuracy": 0.788, "Test F1": 0.770},
    {"Stage": "Tuned Classical ML",      "Model": "TF-IDF + Stacking Ensemble",    "Test Accuracy": 0.805, "Test F1": 0.794},
    {"Stage": "Deep Learning (No GloVe)","Model": "Bidirectional LSTM",            "Test Accuracy": 0.807, "Test F1": 0.792},
    {"Stage": "GloVe (Frozen)",          "Model": "GloVe + BiLSTM (Frozen)",       "Test Accuracy": 0.865, "Test F1": 0.853},
    {"Stage": "GloVe (Fine-tuned)",      "Model": "GloVe + BiGRU",                 "Test Accuracy": 0.877, "Test F1": 0.871},
    {"Stage": "GloVe (Fine-tuned)",      "Model": "GloVe + BiLSTM (Fine-tuned)",   "Test Accuracy": 0.886, "Test F1": 0.880},
    {"Stage": "Best Model",              "Model": "GloVe + Stacked BiLSTM",        "Test Accuracy": 0.889, "Test F1": 0.883},
]
journey_df = pd.DataFrame(all_results)

fig, ax = plt.subplots(figsize=(13, 6))
colors = ["#E5E7EB", "#9CA3AF", "#6B7280", "#4F46E5", "#7C3AED", "#6D28D9", "#F59E0B"]
bars   = ax.barh(journey_df["Model"], journey_df["Test Accuracy"],
                 color=colors, edgecolor="white", height=0.6)

ax.set_xlim(0.70, 0.96)
ax.set_xlabel("Test Accuracy", fontsize=12)
ax.set_title("Accuracy Journey: From Baseline to Best Model", fontsize=14, fontweight="bold")

for bar, val in zip(bars, journey_df["Test Accuracy"]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            str(round(val * 100, 1)) + "%", va="center", fontweight="bold", fontsize=10)

ax.axvline(x=0.889, color="#F59E0B", linestyle="--", lw=2, alpha=0.8, label="Best: 88.9%")
ax.legend()
plt.tight_layout()
plt.savefig("accuracy_journey.png", bbox_inches="tight", dpi=150)
plt.show()

print(journey_df.to_string(index=False))


                   Stage                        Model  Test Accuracy  Test F1
                Baseline TF-IDF + Logistic Regression          0.788    0.770
      Tuned Classical ML   TF-IDF + Stacking Ensemble          0.805    0.794
Deep Learning (No GloVe)           Bidirectional LSTM          0.807    0.792
          GloVe (Frozen)      GloVe + BiLSTM (Frozen)          0.865    0.853
      GloVe (Fine-tuned)                GloVe + BiGRU          0.877    0.871
      GloVe (Fine-tuned)  GloVe + BiLSTM (Fine-tuned)          0.886    0.880
              Best Model       GloVe + Stacked BiLSTM          0.889    0.883


## Section 11: Save All Models

We save the trained models so they can be loaded later by the Streamlit web application for real-time predictions. No need to retrain every time the app starts.


In [44]:
# Save the best deep learning model
os.makedirs("models", exist_ok=True)

stacked_bilstm.save("models/best_dl_model.keras")
print("Deep learning model saved as models/best_dl_model.keras")

with open("models/dl_tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer_g, f)
print("Tokenizer saved as models/dl_tokenizer.pkl")

dl_metadata = {
    "model_name"    : "GloVe + Stacked Bidirectional LSTM",
    "test_accuracy" : round(accuracy_score(y_test_arr, y_pred_stacked), 4),
    "test_f1"       : round(f1_score(y_test_arr, y_pred_stacked), 4),
    "test_auc"      : round(roc_auc_score(y_test_arr, y_proba_stacked), 4),
    "max_len"       : MAX_LEN_G,
    "max_vocab"     : MAX_VOCAB_G,
    "embed_dim"     : EMBED_DIM_G,
}
with open("models/dl_metadata.pkl", "wb") as f:
    pickle.dump(dl_metadata, f)
print("Model metadata saved")
print(dl_metadata)


Deep learning model saved as models/best_dl_model.keras
Tokenizer saved as models/dl_tokenizer.pkl
Model metadata saved
{'model_name': 'GloVe + Stacked Bidirectional LSTM', 'test_accuracy': 0.8918, 'test_f1': 0.8859, 'test_auc': 0.9545, 'max_len': 40, 'max_vocab': 20000, 'embed_dim': 100}


In [45]:
# Also save in TensorFlow SavedModel format for cloud deployment
# The SavedModel format works consistently across different platforms and Python versions
stacked_bilstm.export("models/saved_model")
print("Model also saved in TensorFlow SavedModel format for cloud deployment")

print()
print("All files in the models folder:")
for item in os.listdir("models"):
    item_path = os.path.join("models", item)
    if os.path.isfile(item_path):
        size_mb = round(os.path.getsize(item_path) / (1024 * 1024), 1)
        print(" ", item, "-", size_mb, "MB")
    else:
        print(" ", item, "(folder)")


INFO:tensorflow:Assets written to: models/saved_model\assets


INFO:tensorflow:Assets written to: models/saved_model\assets


Saved artifact at 'models/saved_model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40), dtype=tf.float32, name='keras_tensor_53')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1572887520720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1572887524752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1572887524944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1573087697168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1572887526480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1572887525520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1573087698320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1572887520336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1572887523984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1572887519376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1572887523024: TensorSpec(shape=(), dt

## Section 12: Test the Models on New Headlines

Let us put both models to work and test them on new headlines we have never seen before.


In [46]:
# Define prediction functions for both models
def predict_with_classical_ml(text):
    cleaned  = clean_text(text)
    vec      = tfidf_v2.transform([cleaned])
    prob     = stacking_clf.predict_proba(vec)[0][1]
    label    = "SARCASTIC" if prob > 0.5 else "NOT SARCASTIC"
    return label, round(prob, 3)

def predict_with_glove_bilstm(text):
    cleaned  = clean_text_glove(text)
    seq      = tokenizer_g.texts_to_sequences([cleaned])
    padded   = pad_sequences(seq, maxlen=MAX_LEN_G, padding="post", truncating="post")
    prob     = float(stacked_bilstm.predict(padded, verbose=0)[0][0])
    label    = "SARCASTIC" if prob > 0.5 else "NOT SARCASTIC"
    return label, round(prob, 3)

# Test on a variety of headlines
test_headlines = [
    "Scientists discover water is, in fact, wet",
    "Area man passionate defender of what he imagines constitution to be",
    "Federal Reserve raises interest rates by 0.25 percentage points",
    "Nation's dog owners demand government stop playing fetch with their hearts",
    "Apple announces new iPhone with slightly different camera placement",
    "Study finds people who exercise regularly are insufferably smug about it",
    "Local man unsure why he even bothers",
    "WHO warns of new outbreak in Southeast Asia",
]

print("Testing both models on new headlines")
print("-" * 90)
print(f"{'Headline':<50} | {'Classical ML':^18} | {'GloVe BiLSTM':^18}")
print("-" * 90)

for headline in test_headlines:
    cl_label, cl_prob = predict_with_classical_ml(headline)
    dl_label, dl_prob = predict_with_glove_bilstm(headline)
    short = headline[:48] + ".." if len(headline) > 50 else headline
    print(f"{short:<50} | {cl_prob:.2f} {cl_label[:3]:^5} | {dl_prob:.2f} {dl_label[:3]:^5}")


Testing both models on new headlines
------------------------------------------------------------------------------------------
Headline                                           |    Classical ML    |    GloVe BiLSTM   
------------------------------------------------------------------------------------------
Scientists discover water is, in fact, wet         | 0.95  SAR  | 0.70  SAR 
Area man passionate defender of what he imagines.. | 0.97  SAR  | 1.00  SAR 
Federal Reserve raises interest rates by 0.25 pe.. | 0.33  NOT  | 0.93  SAR 
Nation's dog owners demand government stop playi.. | 0.85  SAR  | 0.96  SAR 
Apple announces new iPhone with slightly differe.. | 0.96  SAR  | 1.00  SAR 
Study finds people who exercise regularly are in.. | 0.96  SAR  | 0.98  SAR 
Local man unsure why he even bothers               | 0.97  SAR  | 1.00  SAR 
WHO warns of new outbreak in Southeast Asia        | 0.19  NOT  | 0.01  NOT 


## Section 13: Summary and Key Takeaways

### Final Results

| Model | Test Accuracy | Test F1 | Test AUC |
|---|---|---|---|
| TF-IDF + Logistic Regression (Baseline) | 78.8% | 0.770 | 0.874 |
| TF-IDF + Stacking Ensemble | 80.5% | 0.794 | 0.889 |
| Bidirectional LSTM (Random Embeddings) | 80.7% | 0.792 | 0.887 |
| GloVe + BiLSTM (Frozen) | 86.5% | 0.853 | 0.944 |
| GloVe + BiGRU | 87.7% | 0.871 | 0.950 |
| GloVe + BiLSTM (Fine-tuned) | 88.6% | 0.880 | 0.955 |
| GloVe + Stacked BiLSTM - BEST | 88.9% | 0.883 | 0.957 |

### What we learned

**1. Preprocessing decisions matter a lot**
We used different preprocessing for classical ML and deep learning. Removing stopwords helped classical ML but hurt deep learning performance because sarcasm signals like "is, in fact" and "what a surprise" are lost.

**2. Pre-trained embeddings are the single biggest improvement**
Random embeddings plateaued at around 80% because 28,000 headlines are not enough to learn word meanings from scratch. GloVe, trained on 6 billion words, immediately pushed accuracy above 86%.

**3. Bidirectional models understand sarcasm better**
Reading the headline from both directions lets the model connect the sarcastic punchline at the end with the serious-sounding setup at the beginning.

**4. Ensemble methods squeeze out extra performance from classical ML**
The Stacking Ensemble improved baseline accuracy from 78.8% to 80.5% by combining the strengths of three different models.

**5. Model format matters for deployment**
Keras H5 files can have version conflicts across platforms. We saved the model in TensorFlow SavedModel format for reliable cloud deployment.

---
Capstone Project 2 | NLP Track | Simran Rani | Data Analyst and Aspiring Data Scientist


## App.py:-

"""
app.py - SarcasmIQ: AI-Powered Sarcasm Detection
Professional Streamlit UI - Cloud Deployment Version
"""

import re, pickle, os, warnings
import numpy as np
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ─────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="SarcasmIQ — AI Sarcasm Detector",
    page_icon="🎭",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ─────────────────────────────────────────────
# NLTK SETUP
# ─────────────────────────────────────────────
@st.cache_resource
def setup_nltk():
    for pkg in ["stopwords", "wordnet", "omw-1.4"]:
        nltk.download(pkg, quiet=True)
    return set(stopwords.words("english")), WordNetLemmatizer()

STOPWORDS, LEMMATIZER = setup_nltk()

# ─────────────────────────────────────────────
# CSS
# ─────────────────────────────────────────────
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&display=swap');
html, body, [class*="css"] { font-family: 'Inter', sans-serif; }
.stApp { background: #0a0a0f; }
#MainMenu, footer, header { visibility: hidden; }
.block-container { padding: 0 2rem 2rem 2rem !important; max-width: 1400px; }
[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #0d0d1a 0%, #0a0a12 100%);
    border-right: 1px solid rgba(139,92,246,0.2);
}
[data-testid="stSidebar"] * { color: #e2e8f0 !important; }
[data-testid="stSelectbox"] > div > div {
    background: rgba(139,92,246,0.1) !important;
    border: 1px solid rgba(139,92,246,0.4) !important;
    border-radius: 10px !important;
    color: #e2e8f0 !important;
}
.stTextArea textarea {
    background: rgba(15,15,30,0.8) !important;
    border: 2px solid rgba(139,92,246,0.3) !important;
    border-radius: 12px !important;
    color: #e2e8f0 !important;
    font-size: 1rem !important;
    padding: 1rem !important;
}
.stTextArea textarea:focus {
    border-color: rgba(139,92,246,0.8) !important;
    box-shadow: 0 0 0 3px rgba(139,92,246,0.15) !important;
}
.stTextArea textarea::placeholder { color: #4a5568 !important; }
.stButton > button {
    width: 100%; border-radius: 12px !important;
    font-weight: 700 !important; font-size: 1rem !important;
    border: none !important; padding: 0.75rem 1.5rem !important;
}
.stButton > button[kind="primary"] {
    background: linear-gradient(135deg, #7c3aed, #4f46e5) !important;
    color: white !important;
    box-shadow: 0 4px 20px rgba(124,58,237,0.4) !important;
}
.stButton > button[kind="secondary"] {
    background: rgba(255,255,255,0.05) !important;
    color: #94a3b8 !important;
    border: 1px solid rgba(255,255,255,0.1) !important;
}
.stTabs [data-baseweb="tab-list"] {
    background: rgba(15,15,30,0.5);
    border-radius: 12px; padding: 4px; gap: 4px;
    border: 1px solid rgba(139,92,246,0.2);
}
.stTabs [data-baseweb="tab"] {
    border-radius: 8px !important; color: #64748b !important;
    font-weight: 600 !important; padding: 0.6rem 1.2rem !important;
}
.stTabs [aria-selected="true"] {
    background: linear-gradient(135deg,#7c3aed,#4f46e5) !important;
    color: white !important;
}
[data-testid="metric-container"] {
    background: rgba(139,92,246,0.08);
    border: 1px solid rgba(139,92,246,0.2);
    border-radius: 12px; padding: 1rem;
}
[data-testid="metric-container"] label { color: #94a3b8 !important; font-size:0.8rem !important; }
[data-testid="metric-container"] [data-testid="metric-value"] { color: #e2e8f0 !important; font-size:1.4rem !important; font-weight:700 !important; }
hr { border-color: rgba(139,92,246,0.2) !important; }
.streamlit-expanderHeader {
    background: rgba(139,92,246,0.08) !important;
    border-radius: 10px !important;
    color: #e2e8f0 !important; font-weight: 600 !important;
}
::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: #0a0a0f; }
::-webkit-scrollbar-thumb { background: rgba(139,92,246,0.4); border-radius: 3px; }
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# TEXT CLEANING
# ─────────────────────────────────────────────
def clean_text_ml(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join([LEMMATIZER.lemmatize(w) for w in text.split()
                     if w not in STOPWORDS and len(w) > 1])

def clean_text_dl(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# ─────────────────────────────────────────────
# LOAD MODELS
# ─────────────────────────────────────────────
@st.cache_resource
def load_ml_model():
    try:
        tfidf = pickle.load(open("models/tfidf_vectorizer.pkl","rb"))
        model = pickle.load(open("models/sarcasm_model.pkl","rb"))
        meta  = pickle.load(open("models/model_metadata.pkl","rb"))
        return tfidf, model, meta, True
    except Exception as e:
        return None, None, {}, False

@st.cache_resource
def load_dl_model():
    try:
        import tensorflow as tf
        import os
        h5_path = "models/best_dl_model.h5"
        # Guard: LFS pointer files are ~130 bytes, real model is ~40MB
        if not os.path.exists(h5_path) or os.path.getsize(h5_path) < 10_000:
            return None, None, {}, False
        model     = tf.keras.models.load_model(h5_path, compile=False)
        tokenizer = pickle.load(open("models/dl_tokenizer.pkl","rb"))
        meta      = pickle.load(open("models/dl_metadata.pkl","rb"))
        return model, tokenizer, meta, True
    except Exception:
        return None, None, {}, False

tfidf, ml_model, ml_meta, ML_OK   = load_ml_model()
dl_model, dl_tokenizer, dl_meta, DL_OK = load_dl_model()

# ─────────────────────────────────────────────
# PREDICT
# ─────────────────────────────────────────────
def predict_ml(text):
    vec = tfidf.transform([clean_text_ml(text)])
    p   = ml_model.predict_proba(vec)[0]
    return float(p[1]), float(p[0])

def predict_dl(text):
    from tensorflow.keras.preprocessing.sequence import pad_sequences
    seq    = dl_tokenizer.texts_to_sequences([clean_text_dl(text)])
    padded = pad_sequences(seq, maxlen=dl_meta.get("max_len",40),
                           padding="post", truncating="post")
    p = float(dl_model.predict(padded, verbose=0)[0][0])
    return p, 1-p

# ─────────────────────────────────────────────
# HEADER
# ─────────────────────────────────────────────
st.markdown("""
<div style="
    background: linear-gradient(135deg, #0d0d1a 0%, #130d2e 50%, #0d0d1a 100%);
    border-bottom: 1px solid rgba(139,92,246,0.3);
    padding: 1.5rem 2rem 1.2rem 2rem;
    margin: -2rem -2rem 2rem -2rem;
    position: relative; overflow: hidden;
">
  <div style="position:absolute;top:-60px;left:10%;width:300px;height:300px;
              background:radial-gradient(circle,rgba(124,58,237,0.15),transparent 70%);pointer-events:none;"></div>
  <div style="position:absolute;top:-40px;right:15%;width:250px;height:250px;
              background:radial-gradient(circle,rgba(79,70,229,0.12),transparent 70%);pointer-events:none;"></div>
  <div style="display:flex;align-items:center;justify-content:space-between;position:relative;">
    <div style="display:flex;align-items:center;gap:1rem;">
      <div style="background:linear-gradient(135deg,#7c3aed,#4f46e5);border-radius:14px;
                  width:48px;height:48px;display:flex;align-items:center;justify-content:center;
                  font-size:1.6rem;box-shadow:0 4px 20px rgba(124,58,237,0.5);">🎭</div>
      <div>
        <div style="font-size:1.6rem;font-weight:900;
                    background:linear-gradient(90deg,#a78bfa,#818cf8);
                    -webkit-background-clip:text;-webkit-text-fill-color:transparent;">SarcasmIQ</div>
        <div style="font-size:0.75rem;color:#64748b;font-weight:500;
                    letter-spacing:1px;text-transform:uppercase;">AI-Powered Sarcasm Detection Engine</div>
      </div>
    </div>
    <div style="display:flex;gap:0.6rem;flex-wrap:wrap;">
      <span style="background:rgba(124,58,237,0.15);border:1px solid rgba(124,58,237,0.4);
                   color:#a78bfa;border-radius:999px;padding:4px 14px;font-size:0.75rem;font-weight:600;">
        🧠 Classical ML · 80.5%
      </span>
      <span style="background:rgba(16,185,129,0.15);border:1px solid rgba(16,185,129,0.4);
                   color:#34d399;border-radius:999px;padding:4px 14px;font-size:0.75rem;font-weight:600;">
        🔬 GloVe + BiLSTM · 88.9%
      </span>
      <span style="background:rgba(245,158,11,0.15);border:1px solid rgba(245,158,11,0.4);
                   color:#fbbf24;border-radius:999px;padding:4px 14px;font-size:0.75rem;font-weight:600;">
        NLP Capstone Project 2
      </span>
    </div>
  </div>
</div>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# SIDEBAR
# ─────────────────────────────────────────────
with st.sidebar:
    st.markdown("""
    <div style="font-size:0.7rem;color:#4a5568;letter-spacing:2px;
                text-transform:uppercase;font-weight:700;margin-bottom:0.8rem;">⚙️ Configuration</div>
    """, unsafe_allow_html=True)

    model_options = []
    if ML_OK: model_options.append("🧠 Classical ML  (80.5%)")
    if DL_OK: model_options.append("🔬 GloVe + Stacked BiLSTM  (88.9%)")
    if not model_options:
        st.error("No models found!")
        st.stop()

    selected  = st.selectbox("Active Model", model_options)
    threshold = st.slider("Sarcasm Threshold", 0.0, 1.0, 0.5, 0.01)

    st.markdown("<hr>", unsafe_allow_html=True)

    # ── Correct metrics per selected model ──
    if "Classical" in selected and ML_OK:
        acc   = ml_meta.get("test_accuracy", 0.805)
        f1    = ml_meta.get("test_f1", 0.794)
        auc   = ml_meta.get("test_auc", 0.889)
        mname = ml_meta.get("model_name", "Stacking Ensemble")
    elif DL_OK:
        acc   = dl_meta.get("test_accuracy", 0.889)
        f1    = dl_meta.get("test_f1", 0.883)
        auc   = dl_meta.get("test_auc", 0.957)
        mname = dl_meta.get("model_name", "GloVe + Stacked BiLSTM")
    else:
        acc, f1, auc, mname = 0, 0, 0, "N/A"

    st.markdown(f"""
    <div style="background:rgba(139,92,246,0.08);border:1px solid rgba(139,92,246,0.2);
                border-radius:12px;padding:1rem;">
      <div style="font-size:0.7rem;color:#64748b;letter-spacing:1px;text-transform:uppercase;
                  font-weight:700;margin-bottom:0.8rem;">📊 Model Performance</div>
      <div style="font-size:0.8rem;color:#a78bfa;font-weight:600;margin-bottom:0.8rem;">{mname}</div>
      <div style="display:grid;grid-template-columns:1fr 1fr;gap:0.6rem;">
        <div style="background:rgba(0,0,0,0.3);border-radius:8px;padding:0.6rem;text-align:center;">
          <div style="font-size:1.1rem;font-weight:800;color:#a78bfa;">{acc:.1%}</div>
          <div style="font-size:0.7rem;color:#64748b;">Accuracy</div>
        </div>
        <div style="background:rgba(0,0,0,0.3);border-radius:8px;padding:0.6rem;text-align:center;">
          <div style="font-size:1.1rem;font-weight:800;color:#34d399;">{f1:.1%}</div>
          <div style="font-size:0.7rem;color:#64748b;">F1 Score</div>
        </div>
        <div style="background:rgba(0,0,0,0.3);border-radius:8px;padding:0.6rem;
                    text-align:center;grid-column:span 2;">
          <div style="font-size:1.1rem;font-weight:800;color:#fbbf24;">{auc:.1%}</div>
          <div style="font-size:0.7rem;color:#64748b;">ROC-AUC</div>
        </div>
      </div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<hr>", unsafe_allow_html=True)

    st.markdown("""
    <div style="font-size:0.7rem;color:#4a5568;letter-spacing:2px;
                text-transform:uppercase;font-weight:700;margin-bottom:0.8rem;">🚀 Accuracy Journey</div>
    """, unsafe_allow_html=True)

    journey = [
        ("TF-IDF + LR",        0.788, "#4a5568"),
        ("TF-IDF + Ensemble",  0.805, "#6366f1"),
        ("BiLSTM (random)",    0.798, "#4a5568"),
        ("GloVe + BiLSTM",     0.865, "#8b5cf6"),
        ("GloVe + BiGRU",      0.877, "#a78bfa"),
        ("GloVe + Stacked ✅", 0.889, "#34d399"),
    ]
    for name, val, color in journey:
        bar_w = int(val * 130)
        st.markdown(f"""
        <div style="margin-bottom:0.5rem;">
          <div style="display:flex;justify-content:space-between;margin-bottom:2px;">
            <span style="font-size:0.72rem;color:#94a3b8;">{name}</span>
            <span style="font-size:0.72rem;color:{color};font-weight:700;">{int(val*100)}%</span>
          </div>
          <div style="background:rgba(255,255,255,0.05);border-radius:999px;height:5px;">
            <div style="width:{bar_w}px;max-width:100%;background:{color};
                        border-radius:999px;height:5px;"></div>
          </div>
        </div>
        """, unsafe_allow_html=True)

    if not DL_OK:
        st.markdown("""
        <div style="background:rgba(245,158,11,0.1);border:1px solid rgba(245,158,11,0.3);
                    border-radius:8px;padding:0.7rem;margin-top:1rem;">
          <div style="font-size:0.75rem;color:#fbbf24;">
            ⚠️ GloVe model loading — using Classical ML mode
          </div>
        </div>
        """, unsafe_allow_html=True)

# ─────────────────────────────────────────────
# TABS
# ─────────────────────────────────────────────
tab1, tab2, tab3 = st.tabs(["🔍  Single Prediction","📁  Batch Analysis","📊  Model Insights"])

# ══ TAB 1 ══
with tab1:
    col_left, col_right = st.columns([1.1, 0.9], gap="large")

    with col_left:
        st.markdown("""
        <div style="margin-bottom:1rem;">
          <div style="font-size:1.4rem;font-weight:800;color:#e2e8f0;margin-bottom:0.3rem;">
            Analyze a Headline
          </div>
          <div style="font-size:0.875rem;color:#64748b;">
            Enter any news headline or sentence to detect sarcasm using AI
          </div>
        </div>
        """, unsafe_allow_html=True)

        user_input = st.text_area(
            "headline", height=130, label_visibility="collapsed",
            placeholder="e.g. 'Scientists discover water is, in fact, wet' ...",
        )

        c1, c2 = st.columns([2,1])
        predict_btn = c1.button("🔮  Detect Sarcasm", type="primary", use_container_width=True)
        c2.button("✕  Clear", type="secondary", use_container_width=True)

        st.markdown("""
        <div style="font-size:0.75rem;color:#4a5568;font-weight:600;
                    text-transform:uppercase;letter-spacing:1px;margin:1rem 0 0.5rem 0;">
          Try a sample
        </div>""", unsafe_allow_html=True)

        samples = [
            ("🎭", "Scientists discover water is, in fact, wet"),
            ("🎭", "Area man passionate defender of what he calls 'my team'"),
            ("📰", "Stock market hits record high amid strong earnings"),
            ("📰", "Local firefighters rescue family from burning building"),
        ]
        for emoji, text in samples:
            if st.button(f"{emoji}  {text}", key=text, use_container_width=True):
                user_input  = text
                predict_btn = True

    with col_right:
        if predict_btn and user_input and user_input.strip():
            with st.spinner("Analyzing..."):
                try:
                    if "Classical" in selected and ML_OK:
                        sarc_p, not_p = predict_ml(user_input)
                    elif DL_OK:
                        sarc_p, not_p = predict_dl(user_input)
                    else:
                        st.error("No model available.")
                        st.stop()

                    is_sarc = sarc_p >= threshold
                    conf    = sarc_p if is_sarc else not_p

                    if is_sarc:
                        st.markdown(f"""
                        <div style="background:linear-gradient(135deg,#7c1d1d,#991b1b,#7c1d1d);
                                    border:1px solid rgba(239,68,68,0.5);border-radius:20px;
                                    padding:2rem;text-align:center;
                                    box-shadow:0 0 40px rgba(239,68,68,0.2);">
                          <div style="font-size:3.5rem;margin-bottom:0.5rem;">😏</div>
                          <div style="font-size:1.8rem;font-weight:900;color:#fca5a5;
                                      letter-spacing:-0.5px;margin-bottom:0.3rem;">Sarcastic!</div>
                          <div style="font-size:0.85rem;color:#f87171;margin-bottom:1.2rem;">
                            This text carries sarcastic intent
                          </div>
                          <div style="background:rgba(0,0,0,0.3);border-radius:12px;
                                      padding:0.8rem;display:inline-block;min-width:160px;">
                            <div style="font-size:2rem;font-weight:900;color:white;">{conf:.1%}</div>
                            <div style="font-size:0.75rem;color:#fca5a5;font-weight:600;">CONFIDENCE</div>
                          </div>
                        </div>""", unsafe_allow_html=True)
                    else:
                        st.markdown(f"""
                        <div style="background:linear-gradient(135deg,#064e3b,#065f46,#064e3b);
                                    border:1px solid rgba(16,185,129,0.5);border-radius:20px;
                                    padding:2rem;text-align:center;
                                    box-shadow:0 0 40px rgba(16,185,129,0.2);">
                          <div style="font-size:3.5rem;margin-bottom:0.5rem;">🙂</div>
                          <div style="font-size:1.8rem;font-weight:900;color:#6ee7b7;
                                      letter-spacing:-0.5px;margin-bottom:0.3rem;">Not Sarcastic</div>
                          <div style="font-size:0.85rem;color:#34d399;margin-bottom:1.2rem;">
                            This text appears genuine
                          </div>
                          <div style="background:rgba(0,0,0,0.3);border-radius:12px;
                                      padding:0.8rem;display:inline-block;min-width:160px;">
                            <div style="font-size:2rem;font-weight:900;color:white;">{conf:.1%}</div>
                            <div style="font-size:0.75rem;color:#6ee7b7;font-weight:600;">CONFIDENCE</div>
                          </div>
                        </div>""", unsafe_allow_html=True)

                    st.write("")
                    st.markdown("""<div style="font-size:0.75rem;color:#4a5568;font-weight:700;
                                text-transform:uppercase;letter-spacing:1px;margin-bottom:0.6rem;">
                                Probability Breakdown</div>""", unsafe_allow_html=True)

                    m1, m2 = st.columns(2)
                    m1.metric("🙂 Not Sarcastic", f"{not_p:.1%}")
                    m2.metric("😏 Sarcastic",     f"{sarc_p:.1%}")

                    st.markdown(f"""
                    <div style="margin:0.8rem 0;">
                      <div style="background:rgba(255,255,255,0.05);border-radius:999px;
                                  height:10px;overflow:hidden;position:relative;">
                        <div style="position:absolute;left:0;top:0;height:100%;
                                    width:{not_p*100:.1f}%;
                                    background:linear-gradient(90deg,#10b981,#34d399);
                                    border-radius:999px 0 0 999px;"></div>
                        <div style="position:absolute;right:0;top:0;height:100%;
                                    width:{sarc_p*100:.1f}%;
                                    background:linear-gradient(90deg,#f87171,#ef4444);
                                    border-radius:0 999px 999px 0;"></div>
                      </div>
                      <div style="display:flex;justify-content:space-between;margin-top:4px;">
                        <span style="font-size:0.72rem;color:#34d399;">← Not Sarcastic</span>
                        <span style="font-size:0.72rem;color:#ef4444;">Sarcastic →</span>
                      </div>
                    </div>""", unsafe_allow_html=True)

                    if ML_OK and DL_OK:
                        with st.expander("🔁 Compare across both models"):
                            rows = []
                            p_ml, _ = predict_ml(user_input)
                            p_dl, _ = predict_dl(user_input)
                            for mname, prob in [("Classical ML (80.5%)", p_ml),
                                                ("GloVe + Stacked BiLSTM (88.9%)", p_dl)]:
                                rows.append({
                                    "Model":        mname,
                                    "Sarcasm %":    f"{prob:.1%}",
                                    "Verdict":      "😏 Sarcastic" if prob>=threshold else "🙂 Not Sarcastic",
                                    "Confidence":   f"{max(prob,1-prob):.1%}",
                                })
                            st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True)

                except Exception as e:
                    st.error(f"Prediction error: {str(e)}")
        else:
            st.markdown("""
            <div style="border:2px dashed rgba(139,92,246,0.2);border-radius:20px;
                        padding:3rem;text-align:center;background:rgba(139,92,246,0.03);">
              <div style="font-size:3rem;margin-bottom:1rem;opacity:0.4;">🎭</div>
              <div style="font-size:1rem;color:#4a5568;font-weight:600;">Your result will appear here</div>
              <div style="font-size:0.8rem;color:#374151;margin-top:0.3rem;">
                Enter a headline and click Detect Sarcasm
              </div>
            </div>""", unsafe_allow_html=True)

# ══ TAB 2 ══
with tab2:
    st.markdown("""
    <div style="margin-bottom:1.5rem;">
      <div style="font-size:1.4rem;font-weight:800;color:#e2e8f0;margin-bottom:0.3rem;">Batch Analysis</div>
      <div style="font-size:0.875rem;color:#64748b;">Upload a CSV to analyze hundreds of headlines at once</div>
    </div>""", unsafe_allow_html=True)

    uploaded = st.file_uploader("Upload CSV", type=["csv"], label_visibility="collapsed")
    if uploaded:
        df_batch = pd.read_csv(uploaded)
        st.dataframe(df_batch.head(), use_container_width=True, hide_index=True)
        col_choice = st.selectbox("Select text column", df_batch.columns)

        if st.button("⚡ Run Batch Analysis", type="primary"):
            prog = st.progress(0, text="Analyzing...")
            probs = []
            for i, text in enumerate(df_batch[col_choice].astype(str)):
                try:
                    if "Classical" in selected and ML_OK:
                        p, _ = predict_ml(text)
                    elif DL_OK:
                        p, _ = predict_dl(text)
                    else: p = 0.5
                    probs.append(p)
                except: probs.append(0.5)
                if i % 10 == 0:
                    prog.progress((i+1)/len(df_batch), text=f"Processing {i+1}/{len(df_batch)}...")

            prog.empty()
            df_batch["sarcasm_probability"] = probs
            df_batch["prediction"] = ["😏 Sarcastic" if p>=threshold else "🙂 Not Sarcastic" for p in probs]
            df_batch["confidence"] = [f"{max(p,1-p):.1%}" for p in probs]

            n_s = sum(1 for p in probs if p >= threshold)
            m1,m2,m3,m4 = st.columns(4)
            m1.metric("Total", f"{len(df_batch):,}")
            m2.metric("😏 Sarcastic", f"{n_s:,}", f"{n_s/len(probs):.1%}")
            m3.metric("🙂 Not Sarcastic", f"{len(probs)-n_s:,}")
            m4.metric("Avg Confidence", f"{np.mean([max(p,1-p) for p in probs]):.1%}")

            st.dataframe(df_batch, use_container_width=True, hide_index=True)
            st.download_button("⬇️ Download Results", df_batch.to_csv(index=False).encode(),
                               "predictions.csv", "text/csv", use_container_width=True)
    else:
        st.markdown("""
        <div style="border:2px dashed rgba(139,92,246,0.2);border-radius:20px;
                    padding:4rem;text-align:center;">
          <div style="font-size:3rem;margin-bottom:1rem;opacity:0.3;">📁</div>
          <div style="font-size:1rem;color:#4a5568;font-weight:600;">Drop your CSV file here</div>
        </div>""", unsafe_allow_html=True)

# ══ TAB 3 ══
with tab3:
    st.markdown("""
    <div style="margin-bottom:1.5rem;">
      <div style="font-size:1.4rem;font-weight:800;color:#e2e8f0;margin-bottom:0.3rem;">Model Insights</div>
      <div style="font-size:0.875rem;color:#64748b;">Complete journey from Classical ML to Deep Learning</div>
    </div>""", unsafe_allow_html=True)

    data = {
        "Model":         ["TF-IDF + LR","TF-IDF + Stacking","BiLSTM (Random)",
                          "GloVe + BiLSTM (Frozen)","GloVe + BiGRU",
                          "GloVe + BiLSTM (Fine-tuned)","GloVe + Stacked BiLSTM ⭐"],
        "Test Accuracy": [0.788,0.805,0.798,0.865,0.878,0.887,0.889],
        "Test F1":       [0.770,0.794,0.793,0.853,0.872,0.880,0.883],
        "Test AUC":      [0.874,0.889,0.887,0.944,0.950,0.955,0.957],
        "Category":      ["Classical ML","Classical ML","Deep Learning",
                          "Deep Learning","Deep Learning","Deep Learning","Deep Learning"],
    }
    df_comp = pd.DataFrame(data)

    fig, axes = plt.subplots(1, 3, figsize=(16,5))
    fig.patch.set_facecolor("#0a0a0f")
    metrics  = ["Test Accuracy","Test F1","Test AUC"]
    palettes = [["#4a5568","#6366f1","#4a5568","#8b5cf6","#a78bfa","#c4b5fd","#34d399"]]*3

    for ax, metric, pal in zip(axes, metrics, palettes):
        ax.set_facecolor("#0d0d1a")
        bars = ax.barh(df_comp["Model"], df_comp[metric], color=pal, height=0.6, edgecolor="none")
        ax.set_xlim(0.7, 1.0)
        ax.set_title(metric, color="#e2e8f0", fontweight="800", fontsize=11, pad=10)
        ax.tick_params(colors="#64748b", labelsize=8)
        for spine in ax.spines.values(): spine.set_visible(False)
        ax.set_xlabel("Score", color="#4a5568", fontsize=9)
        ax.axvline(0.805, color="#6366f1", linestyle="--", linewidth=1, alpha=0.6)
        for bar, val in zip(bars, df_comp[metric]):
            ax.text(val+0.002, bar.get_y()+bar.get_height()/2,
                    f"{val:.1%}", va="center", ha="left", color="#e2e8f0", fontsize=8, fontweight="700")
        ax.grid(axis="x", color="#1e1e2e", linewidth=0.8)

    fig.suptitle("Sarcasm Detection — Model Performance Journey",
                 color="#e2e8f0", fontsize=13, fontweight="900", y=1.01)
    plt.tight_layout()
    st.pyplot(fig)
    plt.close()

    styled = df_comp.copy()
    for col in ["Test Accuracy","Test F1","Test AUC"]:
        styled[col] = styled[col].map("{:.1%}".format)
    st.dataframe(styled, use_container_width=True, hide_index=True)

    st.markdown("""
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:1rem;margin-top:1.5rem;">
      <div style="background:rgba(99,102,241,0.1);border:1px solid rgba(99,102,241,0.3);
                  border-radius:14px;padding:1.2rem;">
        <div style="font-size:1.5rem;margin-bottom:0.5rem;">🧠</div>
        <div style="font-weight:800;color:#818cf8;margin-bottom:0.3rem;">Classical ML</div>
        <div style="font-size:0.8rem;color:#64748b;line-height:1.6;">
          TF-IDF (20K features, 1–3 grams) + Stacking Ensemble (LR+SVM+NB → LR). Fast, interpretable.
        </div>
        <div style="margin-top:0.8rem;font-size:1rem;font-weight:800;color:#818cf8;">80.5%</div>
      </div>
      <div style="background:rgba(16,185,129,0.1);border:1px solid rgba(16,185,129,0.3);
                  border-radius:14px;padding:1.2rem;">
        <div style="font-size:1.5rem;margin-bottom:0.5rem;">🔬</div>
        <div style="font-weight:800;color:#34d399;margin-bottom:0.3rem;">GloVe + Stacked BiLSTM</div>
        <div style="font-size:0.8rem;color:#64748b;line-height:1.6;">
          GloVe 100d (6B tokens) + 3-layer Bidirectional LSTM. Understands word meaning and context.
        </div>
        <div style="margin-top:0.8rem;font-size:1rem;font-weight:800;color:#34d399;">88.9% ⭐</div>
      </div>
      <div style="background:rgba(245,158,11,0.1);border:1px solid rgba(245,158,11,0.3);
                  border-radius:14px;padding:1.2rem;">
        <div style="font-size:1.5rem;margin-bottom:0.5rem;">⚡</div>
        <div style="font-weight:800;color:#fbbf24;margin-bottom:0.3rem;">Transformers (Next Step)</div>
        <div style="font-size:0.8rem;color:#64748b;line-height:1.6;">
          DistilBERT / RoBERTa fine-tuning. Self-attention reads full headline simultaneously.
        </div>
        <div style="margin-top:0.8rem;font-size:1rem;font-weight:800;color:#fbbf24;">90%+ 🚀</div>
      </div>
    </div>""", unsafe_allow_html=True)

st.markdown("""
<div style="margin-top:3rem;border-top:1px solid rgba(139,92,246,0.15);
            padding:1.5rem 0 0.5rem 0;text-align:center;">
  <span style="font-size:0.8rem;color:#374151;">
    SarcasmIQ · Capstone Project 2 · NLP Pipeline: Classical ML → Deep Learning → Transformers
  </span>
</div>""", unsafe_allow_html=True)


## Streamlit url:- https://sarcasmiq---ai-powered-sarcasm-detection-engine-sihag.streamlit.app/


## Github:- https://github.com/Simran280199/SarcasmIQ---AI-Powered-Sarcasm-Detection-Engine/upload/main

In [47]:
1

1